# Update Release Scheduler

## Problem Statement

Given $n$ software updates, each with a:
- `planned_date[i]` — the preferred release day
- `alternate_date[i]` — the fallback release day

**Rules:**
1. Updates must be released in **non-decreasing day order**
2. Each update must use **either** its planned or alternate date
3. You must release all updates

**Goal:** Return the **minimum possible final release day**.

---

## Approach: Greedy + Sort

**Key insight:** If we process updates in order of their planned dates, we can greedily pick the earliest valid date for each one.

A date is **valid** if it's ≥ the last release day used.

**Algorithm:**
1. Sort updates by `planned_date`
2. For each update, prefer `planned_date` if valid, else `alternate_date`
3. If neither is valid, return -1 (infeasible)

**Why greedy works:** Taking the earliest valid date never blocks future updates more than necessary. A later choice could only make the constraint harder to satisfy.

**Complexity:** $O(n \log n)$ for sort, $O(n)$ for the sweep → $O(n \log n)$ total.

In [ ]:
def get_min_days(planned_date, alternate_date):
    """
    Find the minimum final release day given planned and alternate dates.

    Args:
        planned_date: list of preferred release days
        alternate_date: list of fallback release days

    Returns:
        int: minimum final day, or -1 if infeasible

    Time:  O(n log n)
    Space: O(n)
    """
    # Pair each update with both dates, sort by planned date
    updates = sorted(zip(planned_date, alternate_date), key=lambda x: x[0])

    last_day = 0  # last release day used

    for planned, alternate in updates:
        if planned >= last_day:
            last_day = planned        # planned date is valid — take it
        elif alternate >= last_day:
            last_day = alternate      # planned too early — use alternate
        else:
            return -1                 # neither works — infeasible

    return last_day

## Walkthrough

Let's trace through the example step by step.

```
planned   = [3, 7, 4, 9]
alternate = [1, 5, 2, 3]
```

After sorting by planned date:

| Step | planned | alternate | last_day before | Choice | last_day after |
|------|---------|-----------|-----------------|--------|----------------|
| 1 | 3 | 1 | 0 | planned (3 ≥ 0) | 3 |
| 2 | 4 | 2 | 3 | planned (4 ≥ 3) | 4 |
| 3 | 7 | 5 | 4 | planned (7 ≥ 4) | 7 |
| 4 | 9 | 3 | 7 | planned (9 ≥ 7) | 9 |

Result: **9** — we got lucky, all planned dates were valid.

In [ ]:
# Trace through with logging
def get_min_days_verbose(planned_date, alternate_date):
    updates = sorted(zip(planned_date, alternate_date), key=lambda x: x[0])
    last_day = 0

    print(f"{'Step':<6} {'Planned':<10} {'Alternate':<12} {'Last Day (before)':<20} {'Choice':<15} {'Last Day (after)'}")
    print("-" * 80)

    for step, (planned, alternate) in enumerate(updates, 1):
        prev = last_day
        if planned >= last_day:
            last_day = planned
            choice = f'planned ({planned})'
        elif alternate >= last_day:
            last_day = alternate
            choice = f'alternate ({alternate})'
        else:
            print(f"{step:<6} {planned:<10} {alternate:<12} {prev:<20} INFEASIBLE")
            return -1
        print(f"{step:<6} {planned:<10} {alternate:<12} {prev:<20} {choice:<15} {last_day}")

    print(f"\nMinimum final day: {last_day}")
    return last_day


planned   = [3, 7, 4, 9]
alternate = [1, 5, 2, 3]
get_min_days_verbose(planned, alternate)

In [ ]:
# Test cases
test_cases = [
    ([3, 7, 4, 9], [1, 5, 2, 3],  9,  "Basic example"),
    ([1, 2, 3],    [4, 5, 6],     3,  "All planned dates valid"),
    ([5, 5, 5],    [6, 7, 8],     7,  "Duplicate planned dates"),
    ([1],          [10],          1,  "Single update"),
    ([10, 1],      [20, 2],       20, "Alternate needed for second"),
]

print(f"{'Description':<30} {'Expected':<10} {'Got':<10} {'Pass?'}")
print("-" * 60)
for planned, alternate, expected, desc in test_cases:
    result = get_min_days(planned, alternate)
    passed = '✓' if result == expected else '✗'
    print(f"{desc:<30} {expected:<10} {result:<10} {passed}")

---
## Complexity Analysis

| | Complexity | Reason |
|--|------------|--------|
| Time | $O(n \log n)$ | Dominated by sort |
| Space | $O(n)$ | `updates` list stores all pairs |

**Why not $O(n^2)$?** We don't compare every pair — we sort once and sweep linearly.

**Edge cases handled:**
- Duplicate planned dates → alternate dates resolve ties
- Infeasible inputs → returns -1
- Single update → trivially returns planned date